In [ ]:
# Lane Detection and Trajectory Planning for ADAS

import cv2
import numpy as np

# 1. Read input road image
image = cv2.imread("requirements/road.jpg")

if image is None:
	print("Image not found!")
	raise SystemExit

# Keep a copy of original image
output = image.copy()

# 2. Convert image to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# 3. Detect edges using Canny
edges = cv2.Canny(gray, 50, 150)

# 4. Define Region of Interest (road area)
height, width = image.shape[:2]

mask = np.zeros_like(edges)

roi_points = np.array([[
	(0, height),
	(width, height),
	(int(width * 0.60), int(height * 0.55)),
	(int(width * 0.40), int(height * 0.55))
]])

cv2.fillPoly(mask, roi_points, 255)

roi = cv2.bitwise_and(edges, mask)

# 5. Detect lane lines using Hough Transform
lines = cv2.HoughLinesP(
	roi,
	1,
	np.pi / 180,
	threshold=50,
	minLineLength=50,
	maxLineGap=100
)

left_lines = []
right_lines = []

if lines is not None:
	for line in lines:
		line = np.array(line).flatten()

		if len(line) != 4:
			continue

		x1, y1, x2, y2 = line

		# Calculate slope
		if x2 - x1 == 0:
			continue

		slope = (y2 - y1) / (x2 - x1)

		# Separate left and right lane lines
		if slope < -0.5:
			left_lines.append((x1, y1, x2, y2))
		elif slope > 0.5:
			right_lines.append((x1, y1, x2, y2))

# Function to calculate average lane line
def average_line(lines):
	if len(lines) == 0:
		return None

	x1_values = []
	y1_values = []
	x2_values = []
	y2_values = []

	for x1, y1, x2, y2 in lines:
		x1_values.append(x1)
		y1_values.append(y1)
		x2_values.append(x2)
		y2_values.append(y2)

	return (
		int(np.mean(x1_values)),
		int(np.mean(y1_values)),
		int(np.mean(x2_values)),
		int(np.mean(y2_values))
	)

left_lane = average_line(left_lines)
right_lane = average_line(right_lines)

# 6. Draw detected lane lines
if left_lane is not None:
	x1, y1, x2, y2 = left_lane
	cv2.line(output, (x1, y1), (x2, y2), (0, 255, 0), 5)

if right_lane is not None:
	x1, y1, x2, y2 = right_lane
	cv2.line(output, (x1, y1), (x2, y2), (0, 255, 0), 5)

# 7. Calculate lane center
if left_lane is not None and right_lane is not None:
	# Use bottom points of lane lines
	left_x = left_lane[2]
	right_x = right_lane[2]

	lane_center = int((left_x + right_x) / 2)

	# Vehicle center
	vehicle_center = width // 2

	# 8. Calculate trajectory error
	error = lane_center - vehicle_center

	# Draw desired trajectory
	cv2.line(
		output,
		(lane_center, height),
		(lane_center, int(height * 0.55)),
		(255, 0, 0),
		5
	)

	# Draw vehicle center
	cv2.circle(
		output,
		(vehicle_center, height - 50),
		10,
		(0, 0, 255),
		-1
	)

	# 9. Basic motion control
	if error > 30:
		steering = "STEER RIGHT"
	elif error < -30:
		steering = "STEER LEFT"
	else:
		steering = "GO STRAIGHT"

	# Display information
	cv2.putText(
		output,
		"Trajectory Center: " + str(lane_center),
		(30, 40),
		cv2.FONT_HERSHEY_SIMPLEX,
		0.8,
		(255, 255, 255),
		2
	)

	cv2.putText(
		output,
		"Error: " + str(error),
		(30, 75),
		cv2.FONT_HERSHEY_SIMPLEX,
		0.8,
		(255, 255, 255),
		2
	)

	cv2.putText(
		output,
		steering,
		(30, 115),
		cv2.FONT_HERSHEY_SIMPLEX,
		1,
		(0, 255, 255),
		3
	)

	print("Lane Center:", lane_center)
	print("Vehicle Center:", vehicle_center)
	print("Tracking Error:", error)
	print("Steering Action:", steering)
else:
	print("Lane lines could not be detected.")

# 10. Display results
cv2.imshow("Original Image", image)
cv2.imshow("Edges", edges)
cv2.imshow("Trajectory and Motion Control", output)

cv2.waitKey(0)
cv2.destroyAllWindows()

Lane Center: 562
Vehicle Center: 600
Tracking Error: -38
Steering Action: STEER LEFT


: 